<a href="https://colab.research.google.com/github/osacr6/SC707_Proyecto_G6/blob/Daniel/SC707_Proyecto_G6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Grupo 6 · SC-707
## Big Data - Universidad Fidélitas
## Proyecto final - Avance 2
#### Perfiles de los cantones de Costa Rica según sus pymes activas

- Cambios en la cantidad de pymes del cantón entre 2020 y 2026 (Crecimiento).
- PCT de las pymes que siguen en el registro desde el 2020 (Permanencia)

**Datos:** cada registro es una pyme en un año y un cantón (lista de pymes activas del MEIC, enero de 2020 a 2026), más la población de cada cantón.

**Subir todos los archivos en `MyDrive/sc-707-pymes`**.

# 1 - Datos

In [18]:
from google.colab import drive
drive.mount('/content/drive')


import os, re, unicodedata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

# Subir archivos a Drive
DATA_DIR = BASE_DIR = '/content/drive/MyDrive/sc-707-pymes'
AÑOS = range(2020, 2027)

# Población https://www.geo-ref.net/sp/cri.htm
POBLACION = pd.read_csv(os.path.join(BASE_DIR, 'poblacion_cantones.csv'))
POBLACION = (POBLACION.groupby(['PROVINCIA', 'CANTON_MEIC'], as_index=False)[['POBLACION', 'AREA_KM2']].sum()
          .rename(columns={'CANTON_MEIC': 'CANTON'}))
POBLACION['DENSIDAD'] = POBLACION['POBLACION'] / POBLACION['AREA_KM2']

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.1 Unir archivos

- Cada año trae columnas con otros nombres, en otro orden o faltan.
- Se conservan las que existen en todos los años

In [6]:
COLUMNAS = [
  ('IDENTIFICACI', 'ID'),
  ('TAMA', 'TAMANO'),
  ('PROVINCIA', 'PROVINCIA'),
  ('CANT', 'CANTON'),
  ('REGI', 'REGION'),
  ('ACTIVIDAD CIIU', 'CIIU'),
  ('ACTIVIDAD PRIMARIA', 'CIIU')
]

# Normaliza texto: todo mayúscula, sin tildes, sin espacios extra
def _norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return re.sub(r'\s+', ' ', s).strip().upper()

def renombrar(cols):
    nuevo = {}
    for c in cols:
        for prefijo, nombre in COLUMNAS:
            if _norm(c).startswith(prefijo) and nombre not in nuevo.values():
                nuevo[c] = nombre
                break
    return nuevo

reporte, partes = [], []

for anio in AÑOS:
    crudo = pd.read_csv(os.path.join(DATA_DIR, f'{anio}-01.csv'), encoding='latin-1', dtype=str)
    crudo = crudo.loc[:, ~crudo.columns.str.startswith('Unnamed')].dropna(how='all')
    mapa = renombrar(crudo.columns)
    t = crudo.rename(columns=mapa)[list(mapa.values())].copy()
    t['ANIO'] = anio
    reporte.append({'Año': anio, 'Registros': len(t), 'Columnas': crudo.shape[1]})
    partes.append(t)

reporte = pd.DataFrame(reporte)
reporte


,Año,Registros,Columnas
0,2020,19923,12
1,2021,24038,11
2,2022,25691,11
3,2023,21644,12
4,2024,25606,12
5,2025,25661,9
6,2026,31757,13


### 1.1.1 Raw data

In [7]:

DATA = pd.concat(partes, ignore_index=True)

for c in ['ID', 'TAMANO', 'PROVINCIA', 'CANTON', 'REGION', 'CIIU']:
    DATA[c] = DATA[c].astype('string').str.strip()

DATA.head()

,ID,TAMANO,PROVINCIA,CANTON,REGION,CIIU,ANIO
0,101194919,Micro,Limón,Pococí,Huetar Atlántica,7110-Actividades de arquitectura e ingeniería ...,2020
1,102250305,Micro,San José,San José,Central Oriental,4751-Venta al por menor de textiles en almacen...,2020
2,102510557,Micro,San José,San José,Central Oriental,6910-Actividades jurídicas.,2020
3,102640566,Micro,San José,Moravia,Central Oriental,"4771-Venta al por menor de productos textiles,...",2020
4,102700492,Micro,San José,San José,Central Oriental,4520-Mantenimiento y reparación de vehículos a...,2020


## 1.2 Limpieza

In [9]:
reporte = {}


# Identificación limpia: mayúsculas, sin espacios ni guiones; si es solo números, sin ceros a la izquierda
ide = DATA['ID'].fillna('').str.upper().str.replace(r'[^0-9A-Z]', '', regex=True)
DATA['ID'] = ide.where(~ide.str.fullmatch(r'\d+'), ide.str.lstrip('0'))
DATA = DATA[DATA['ID'] != '']

dup = DATA.duplicated(subset=['ANIO', 'ID'])
reporte['Identificación repetida'] = DATA[dup].groupby('ANIO').size()
DATA = DATA[~dup]

DATA['CIIU_COD'] = DATA['CIIU'].str.extract(r'^\s*(\d{4})', expand=False)
reporte['Sin código CIIU'] = DATA[DATA['CIIU_COD'].isna()].groupby('ANIO').size()

pd.DataFrame(reporte).fillna(0).astype(int).rename_axis('Año')

,Identificación repetida,Sin código CIIU
Año,,
2020,82,12
2021,5,18
2022,3,15
2023,0,9
2024,1,15
2025,1,16
2026,2,30


#2 - Análisis exploratorio

## 2.1 Tamaño de las Pymes


In [10]:
(pd.crosstab(DATA['ANIO'], DATA['TAMANO'], normalize='index') * 100).round(1)

TAMANO,Mediana,Micro,Pequeña
ANIO,,,
2020,4.3,77.3,18.5
2021,3.8,79.6,16.6
2022,2.8,84.0,13.3
2023,1.8,88.2,10.0
2024,1.2,89.9,8.9
2025,0.4,90.1,9.4
2026,0.4,90.9,8.7


## 2.2 Actividad económica canton



# Sección nueva

In [11]:
# Cada pyme trae un código CIIU de 4 dígitos, por ejemplo 5610 - Actividades de restaurantes…
SECCIONES_CIIU = [
    ( 1,  3, 'A', 'Agropecuario'),
    ( 5,  9, 'B', 'Minas'),
    (10, 33, 'C', 'Manufactura'),
    (35, 35, 'D', 'Electricidad'),
    (36, 39, 'E', 'Agua y residuos'),
    (41, 43, 'F', 'Construcción'),
    (45, 47, 'G', 'Comercio'),
    (49, 53, 'H', 'Transporte'),
    (55, 56, 'I', 'Alojamiento y comidas'),
    (58, 63, 'J', 'Información y comunicación'),
    (64, 66, 'K', 'Financieras'),
    (68, 68, 'L', 'Inmobiliarias'),
    (69, 75, 'M', 'Profesionales'),
    (77, 82, 'N', 'Administrativos'),
    (84, 84, 'O', 'Administración pública'),
    (85, 85, 'P', 'Enseñanza'),
    (86, 88, 'Q', 'Salud'),
    (90, 93, 'R', 'Artes y recreación'),
    (94, 96, 'S', 'Otros servicios'),
    (97, 98, 'T', 'Hogares'),
    (99, 99, 'U', 'Extraterritoriales'),
]
NOMBRE_ACTIVIDADES = {s: n for _, _, s, n in SECCIONES_CIIU}

def seccion(cod):
    if pd.isna(cod): return 'Sin código'
    d = int(str(cod)[:2])
    return next((s for a, b, s, _ in SECCIONES_CIIU if a <= d <= b), 'Sin código')

DATA['SECCION'] = DATA['CIIU_COD'].map(seccion)
comp_anio = (pd.crosstab(DATA['SECCION'], DATA['ANIO'], normalize='columns') * 100).round(1)
comp_anio.index = [f'{s} · {NOMBRE_ACTIVIDADES.get(s, s)}' for s in comp_anio.index]
comp_anio.sort_values(2026, ascending=False).head(10)

ANIO,2020,2021,2022,2023,2024,2025,2026
G · Comercio,34.0,33.2,32.2,32.1,31.7,30.8,29.6
M · Profesionales,12.2,12.4,13.0,13.7,13.6,13.6,13.7
C · Manufactura,10.0,10.2,10.8,11.0,11.3,10.9,10.7
I · Alojamiento y comidas,8.6,8.8,8.6,8.1,8.6,9.3,9.4
Q · Salud,7.2,7.1,6.8,7.2,6.9,6.8,7.0
N · Administrativos,5.4,5.6,5.7,5.6,5.4,5.2,5.5
F · Construcción,3.8,4.3,4.7,4.9,5.1,5.3,5.4
H · Transporte,4.2,4.3,4.4,4.0,4.4,4.7,4.7
S · Otros servicios,6.1,5.6,5.2,5.1,4.2,4.3,4.5
J · Información y comunicación,2.8,3.1,3.3,3.1,3.2,3.1,3.2


In [36]:
# CIIU con mayor peso en el total de pymes
ACTIVIDADES_PRINCIPALES = {
    'G': 'PCT_COMERCIO',
    'M': 'PCT_PROFESIONALES',
    'C': 'PCT_MANUFACTURA',
    'I': 'PCT_ALOJ_COMIDAS',
    'Q': 'PCT_SALUD',
    'F': 'PCT_CONSTRUCCION',
}
PCT_ACTIVIDADES = list(ACTIVIDADES_PRINCIPALES.values())
PROVINCIA_CANTON = ['PROVINCIA', 'CANTON']

s20 = DATA[DATA['ANIO'] == 2020]
s26 = DATA[DATA['ANIO'] == 2026]

# PYMES
cant_pymes = s26.groupby(PROVINCIA_CANTON).size().rename('PYMES_2026').to_frame()
cant_pymes['PYMES_2020'] = s20.groupby(PROVINCIA_CANTON).size()
pct_pymes_26 = pd.crosstab([
    s26['PROVINCIA'],
    s26['CANTON']
    ], s26['SECCION'], normalize='index') * 100

#print(cant_pymes)
#print(pct_pymes_26)

# PCT de actividades segun 2026
for i, actividad in ACTIVIDADES_PRINCIPALES.items():
    cant_pymes[actividad] = pct_pymes_26[i] if i in pct_pymes_26 else 0.0

cant_pymes.head()

PYMES_2026  PYMES_2020  PCT_COMERCIO  \
PROVINCIA CANTON                                              
Alajuela  Alajuela           1920        1060     30.885417   
          Alfaro Ruiz          92          70     18.478261   
          Atenas              173          58     31.213873   
          Grecia              509         342     30.648330   
          Guatuso              35          17     48.571429   

                       PCT_PROFESIONALES  PCT_MANUFACTURA  PCT_ALOJ_COMIDAS  \
PROVINCIA CANTON                                                              
Alajuela  Alajuela             11.927083        10.781250          8.593750   
          Alfaro Ruiz          14.130435        35.869565          6.521739   
          Atenas               12.138728        12.138728          6.358382   
          Grecia               12.966601        11.787819          5.500982   
          Guatuso               5.714286         5.714286         11.428571   

                       PCT_SALUD  PCT_CONSTRUCCION  
PROVINCIA CANTON                                    
Alajuela  Alajuela      4.687500          6.666667  
          Alfaro Ruiz   5.434783          1.086957  
          Atenas       11.560694          6.936416  
          Grecia        6.876228          7.662083  
          Guatuso       0.000000          2.857143

## 2.3 Crecimiento


In [30]:
cant_pymes['CRECIMIENTO'] = np.log(cant_pymes['PYMES_2026'] / cant_pymes['PYMES_2020'])

cant_pymes.head()

PYMES_2026  PYMES_2020  PCT_COMERCIO  \
PROVINCIA CANTON                                              
Alajuela  Alajuela           1920        1060     30.885417   
          Alfaro Ruiz          92          70     18.478261   
          Atenas              173          58     31.213873   
          Grecia              509         342     30.648330   
          Guatuso              35          17     48.571429   

                       PCT_PROFESIONALES  PCT_MANUFACTURA  PCT_ALOJ_COMIDAS  \
PROVINCIA CANTON                                                              
Alajuela  Alajuela             11.927083        10.781250          8.593750   
          Alfaro Ruiz          14.130435        35.869565          6.521739   
          Atenas               12.138728        12.138728          6.358382   
          Grecia               12.966601        11.787819          5.500982   
          Guatuso               5.714286         5.714286         11.428571   

                       PCT_SALUD  PCT_CONSTRUCCION  CRECIMIENTO  
PROVINCIA CANTON                                                 
Alajuela  Alajuela      4.687500          6.666667     0.594056  
          Alfaro Ruiz   5.434783          1.086957     0.273293  
          Atenas       11.560694          6.936416     1.092849  
          Grecia        6.876228          7.662083     0.397637  
          Guatuso       0.000000          2.857143     0.722135

## 2.4 Permanencia



In [37]:
ids_2026 = set(s26['ID'])
cant_pymes['PERMANENCIA'] = s20.assign(SIGUE=s20['ID'].isin(ids_2026)).groupby(PROVINCIA_CANTON)['SIGUE'].mean() * 100

cant = cant_pymes.reset_index()
cant = cant.merge(POBLACION, on=PROVINCIA_CANTON, how='left')
cant.head()


,PROVINCIA,CANTON,PYMES_2026,PYMES_2020,PCT_COMERCIO,PCT_PROFESIONALES,PCT_MANUFACTURA,PCT_ALOJ_COMIDAS,PCT_SALUD,PCT_CONSTRUCCION,PERMANENCIA,POBLACION,AREA_KM2,DENSIDAD
0,Alajuela,Alajuela,1920,1060,30.885417,11.927083,10.781250,8.593750,4.687500,6.666667,40.754717,322143,758.3,424.822630
1,Alajuela,Alfaro Ruiz,92,70,18.478261,14.130435,35.869565,6.521739,5.434783,1.086957,20.000000,15153,155.1,97.698259
2,Alajuela,Atenas,173,58,31.213873,12.138728,12.138728,6.358382,11.560694,6.936416,34.482759,30407,127.2,239.048742
3,Alajuela,Grecia,509,342,30.648330,12.966601,11.787819,5.500982,6.876228,7.662083,38.011696,90644,396.0,228.898990
4,Alajuela,Guatuso,35,17,48.571429,5.714286,5.714286,11.428571,0.000000,2.857143,35.294118,18316,758.3,24.154029


## 2.5 Tabla de frecuencia

In [ ]:
# ['mean', 'median', 'std', 'min', 'max']

# 3 Histogramas

## 3.1 Concentración de las pymes

## 3.2 Evolución y permanencia por región

# 4 Valores atípicos

In [ ]:
# rango intercuartílico

 # 5 K-means